[Guide Index](README.md)


# 06a Serialization Formats

`StarLayerGraph.parse()` / `.serialize()` read and write the RDF formats Oxigraph supports, all of them carrying RDF 1.2 content (triple terms, reifiers, language direction). `StarLayerGraph` is the working name of the Oxigraph-backed graph class that becomes `StarLayerGraph` at the switch (see [the Oxigraph-backed graph guide](02d-oxigraph-wrapper.ipynb)); the format names are the plain ones, with the old `...12` spellings kept as aliases:

| `format=` | Aliases | Notes |
|---|---|---|
| `turtle` | `turtle12`, `ttl`, `n3` | Turtle 1.2. `n3` is read and written as Turtle. `pretty=True` gives the grouped, folded layout |
| `nt` | `nt12`, `ntriples` | N-Triples 1.2 |
| `nq` | `nq12`, `nquads` | N-Quads 1.2 (quad format) |
| `trig` | `trig12` | TriG 1.2 (quad format) |
| `xml` | `rdfxml12`, `rdf-xml` | RDF/XML 1.2 |
| `json-ld` | `jsonld` | works for plain RDF; Oxigraph cannot yet write RDF 1.2 content as JSON-LD |

Two formats of the old class are dropped: **`trix12`** (TriX; there is no replacement) and **`longturtle12`** (the flat one-triple-per-line layout is what plain `turtle` writes now, and `serialize(format="turtle", pretty=True)` is the readable layout). Asking for either raises `ValueError`. Manchester Syntax (`format="manchester"`) is separate; see [the Manchester guide](05a-manchester-syntax.ipynb).

This guide is about the formats themselves; see the [Graphs guide](02-graphs.ipynb) for the semantics being serialized, and the [datasets guide](02a-graphs-datasets.ipynb) for the dataset-capable formats.

## How to run this notebook

This guide assumes StarLayer has been pip installed — see [Getting Started](01-getting-started.ipynb). Run cells from top to bottom; later cells reuse variables from earlier ones.


In [1]:
from rdflib import Namespace
from starlayer.graph.oxgraph import StarLayerGraph, StarLayerDataset
from starlayer.graph.compare import isomorphic
from starlayer.graph import rdfc_native

EX = Namespace("http://example.org/")
QUAD_FORMATS = {"nq", "trig"}

# helper output function
def test_roundtrip(graph, fmt):
    """Serialize `graph` as `fmt`, reparse it, and compare to the original.

    Prints a clean match, or - if the round trip differs - the exact
    triples present on only one side or the other. A quad format keeps the
    graph's name, so it is read back into an StarLayerDataset; a triple format
    is read back into an StarLayerGraph.
    """
    text = graph.serialize(format=fmt)
    if fmt in QUAD_FORMATS:
        roundtripped = StarLayerDataset()
        roundtripped.parse(data=text, format=fmt)
        same = rdfc_native.isomorphic(graph.store, roundtripped.store)  # graph names count
    else:
        roundtripped = StarLayerGraph()
        roundtripped.parse(data=text, format=fmt)
        same = isomorphic(graph, roundtripped)
    if same:
        print(f"{fmt}: matches the original.")
        return
    print(f"{fmt}: differs from the original.")
    for t in sorted(set(graph) - set(roundtripped), key=str):
        print("  only in original:   ", t)
    for t in sorted(set(roundtripped) - set(graph), key=str):
        print("  only in round-trip: ", t)

## 1. `turtle` — the reference example

**Turtle 1.2: the human-friendly, prefix-based syntax**, with all the RDF 1.2 shorthand (`<<( )>>`, `<< >>`, `{| |}`, `~`) — the format used throughout the rest of this guide set. Oxigraph's own Turtle writer streams one statement at a time (section 2.1 shows it); `pretty=True` switches to StarLayer's pretty writer, which groups a subject's predicates with `;`, uses `a`, folds reifiers into `{| |}` / `~` and writes only the prefixes it uses. The cell below prints that layout.

The graph below is reused for every format, so each one can be compared directly against the same content. It's constructed with an explicit `identifier=EX.main` (rather than leaving it as the default graph) so the dataset-capable formats later in this guide (`nq`, `trig`) have a real graph name to show — that identifier is a property of the `StarLayerGraph` object itself, set in Python, not something expressed anywhere in the Turtle text below. The graph lives in an Oxigraph store; a fresh `StarLayerGraph` has no prefixes bound, so the cell binds `ex:` (the document's own `@prefix` lines add `rdf:` and `xsd:` when it is parsed).

In [2]:
# identifier=EX.main gives this graph a real name (rather than an
# auto-generated one) - see the markdown above for why.
g_parsed = StarLayerGraph(identifier=EX.main)
g_parsed.bind("ex", EX)
# parse() reads RDF 1.2 terms (triple terms, reifiers, annotation blocks) in every format below.
g_parsed.parse(data='''
    @prefix ex: <http://example.org/> .
    @prefix rdf: <http://www.w3.org/1999/02/22-rdf-syntax-ns#> .
    @prefix xsd: <http://www.w3.org/2001/XMLSchema#> .

    # language-direction literals
    ex:listing_en ex:description "hello"@en--ltr .
    ex:listing_ar ex:description "مرحبا"@ar--rtl .

    # canonical reification with rdf:reifies
    ex:claim rdf:reifies <<( ex:alice ex:worksFor ex:AcmeCorp )>> ;
      ex:source ex:HRSystem ;
      ex:confidence "high" .

    # reifying shorthand << s p o >>: creates an anonymous reifier
    << ex:alice ex:worksFor ex:AcmeCorp >> ex:source ex:HRSystem .

    # reifying shorthand with a named reifier: << s p o ~ id >>
    << ex:alice ex:worksFor ex:AcmeCorp ~ ex:id >> ex:source ex:HRSystem ;
      ex:reportedDate "2026-01-01"^^xsd:date .

    # anonymous inline annotation block
    ex:alice ex:likes ex:ProductABC {| ex:since "2020" ; ex:source ex:CRM |} .

    # named reifier with annotations
    ex:alice ex:mentions ex:GlobalTech ~ ex:stmt1 {| ex:confidence "0.9" ; ex:source ex:AuditSystem |} .

    # named reifier without annotation block
    ex:alice ex:worksWith ex:SalesTeam ~ ex:stmt2 .

    # an additional quoted triple term reused in the query examples elsewhere
    ex:AuditSystem ex:reported <<( ex:alice ex:worksFor ex:AcmeCorp )>> .
''', format='turtle')

# pretty=True is the grouped, folded layout; the default writer is flat (section 2.1).
print(g_parsed.serialize(format='turtle', pretty=True))

@version "1.2" .
@prefix ex: <http://example.org/> .
@prefix rdf: <http://www.w3.org/1999/02/22-rdf-syntax-ns#> .
@prefix xsd: <http://www.w3.org/2001/XMLSchema#> .

ex:AuditSystem ex:reported <<( ex:alice ex:worksFor ex:AcmeCorp )>> .

ex:alice ex:likes ex:ProductABC {| ex:since "2020" ; ex:source ex:CRM |} ;
    ex:mentions ex:GlobalTech ~ ex:stmt1 {| ex:confidence "0.9" ; ex:source ex:AuditSystem |} ;
    ex:worksWith ex:SalesTeam ~ ex:stmt2 .

ex:listing_ar ex:description "مرحبا"@ar--rtl .

ex:listing_en ex:description "hello"@en--ltr .

<< ex:alice ex:worksFor ex:AcmeCorp ~ ex:claim >> ex:confidence "high" ;
    ex:source ex:HRSystem .

<< ex:alice ex:worksFor ex:AcmeCorp ~ ex:id >> ex:reportedDate "2026-01-01"^^xsd:date ;
    ex:source ex:HRSystem .

<< ex:alice ex:worksFor ex:AcmeCorp >> ex:source ex:HRSystem .



## 2. The other formats

`turtle` above is one of six formats `parse()`/`serialize()` support. Each subsection below covers one format: what it is, and what the reference graph looks like serialized into it. TriX is gone (section 2.5).

### 2.1 Flat Turtle (formerly `longturtle12`)

**Turtle 1.2 as Oxigraph's own writer produces it: flat and fully expanded.** This is what plain `serialize(format="turtle")` returns, and it is the layout the old `longturtle12` stood for. Every statement is on its own line, no shorthand blocks (`{| |}`/`~`/`<< >>` are all expanded to plain `rdf:reifies` triples) — trades compactness for a form where structure is unambiguous at a glance. `longturtle` itself (rdflib's name for a different layout, one object per line) is not offered; asking for `longturtle12` or `longturtle` raises `ValueError`.

In [3]:
print(g_parsed.serialize(format="turtle"))

@version "1.2" .
@prefix rdf: <http://www.w3.org/1999/02/22-rdf-syntax-ns#> .
@prefix xsd: <http://www.w3.org/2001/XMLSchema#> .
@prefix ex: <http://example.org/> .
ex:AuditSystem ex:reported <<( ex:alice ex:worksFor ex:AcmeCorp )>> .
ex:stmt2 rdf:reifies <<( ex:alice ex:worksWith ex:SalesTeam )>> .
ex:alice ex:worksWith ex:SalesTeam .
ex:stmt1 ex:source ex:AuditSystem ;
	ex:confidence "0.9" ;
	rdf:reifies <<( ex:alice ex:mentions ex:GlobalTech )>> .
ex:alice ex:mentions ex:GlobalTech .
_:f376e376c533a70f781b4ee2787a5608 ex:source ex:CRM ;
	ex:since "2020" ;
	rdf:reifies <<( ex:alice ex:likes ex:ProductABC )>> .
ex:alice ex:likes ex:ProductABC .
ex:id ex:reportedDate "2026-01-01"^^xsd:date ;
	ex:source ex:HRSystem ;
	rdf:reifies <<( ex:alice ex:worksFor ex:AcmeCorp )>> .
_:d2c2c3da891c9b220417c7b76f84b424 ex:source ex:HRSystem ;
	rdf:reifies <<( ex:alice ex:worksFor ex:AcmeCorp )>> .
ex:claim ex:confidence "high" ;
	ex:source ex:HRSystem ;
	rdf:reifies <<( ex:alice ex:worksFor ex:AcmeC

In [4]:
test_roundtrip(g_parsed, "turtle")

turtle: matches the original.


### 2.2 `nt` (alias `nt12`)

**N-Triples 1.2: the flattest serialization.** One full triple per line, absolute IRIs only (no prefixes), no shorthand. Easy for systems to stream line by line; not meant to be hand-written.

In [5]:
print(g_parsed.serialize(format="nt"))

VERSION "1.2"
<http://example.org/AuditSystem> <http://example.org/reported> <<( <http://example.org/alice> <http://example.org/worksFor> <http://example.org/AcmeCorp> )>> .
<http://example.org/stmt2> <http://www.w3.org/1999/02/22-rdf-syntax-ns#reifies> <<( <http://example.org/alice> <http://example.org/worksWith> <http://example.org/SalesTeam> )>> .
<http://example.org/alice> <http://example.org/worksWith> <http://example.org/SalesTeam> .
<http://example.org/stmt1> <http://example.org/source> <http://example.org/AuditSystem> .
<http://example.org/stmt1> <http://example.org/confidence> "0.9" .
<http://example.org/stmt1> <http://www.w3.org/1999/02/22-rdf-syntax-ns#reifies> <<( <http://example.org/alice> <http://example.org/mentions> <http://example.org/GlobalTech> )>> .
<http://example.org/alice> <http://example.org/mentions> <http://example.org/GlobalTech> .
_:f376e376c533a70f781b4ee2787a5608 <http://example.org/source> <http://example.org/CRM> .
_:f376e376c533a70f781b4ee2787a5608 <htt

In [6]:
test_roundtrip(g_parsed, "nt")

nt: matches the original.


### 2.3 `nq` (alias `nq12`)

**N-Quads 1.2: `nt` plus a graph name.** Same one-line-per-fact format, with a fourth term naming which graph each quad belongs to. Since it's one graph, every line below repeats the same graph name; a dataset with more than one named graph would show a different name per line — see the [datasets guide](02a-graphs-datasets.ipynb). A document with named graphs can only be read into an `StarLayerDataset`; parsing it into an `StarLayerGraph` raises `ValueError`, which is why the round-trip helper reads quad formats into a dataset.

In [7]:
print(g_parsed.serialize(format="nq"))

VERSION "1.2"
<http://example.org/AuditSystem> <http://example.org/reported> <<( <http://example.org/alice> <http://example.org/worksFor> <http://example.org/AcmeCorp> )>> <http://example.org/main> .
<http://example.org/stmt2> <http://www.w3.org/1999/02/22-rdf-syntax-ns#reifies> <<( <http://example.org/alice> <http://example.org/worksWith> <http://example.org/SalesTeam> )>> <http://example.org/main> .
<http://example.org/alice> <http://example.org/worksWith> <http://example.org/SalesTeam> <http://example.org/main> .
<http://example.org/stmt1> <http://example.org/source> <http://example.org/AuditSystem> <http://example.org/main> .
<http://example.org/stmt1> <http://example.org/confidence> "0.9" <http://example.org/main> .
<http://example.org/stmt1> <http://www.w3.org/1999/02/22-rdf-syntax-ns#reifies> <<( <http://example.org/alice> <http://example.org/mentions> <http://example.org/GlobalTech> )>> <http://example.org/main> .
<http://example.org/alice> <http://example.org/mentions> <http:/

In [8]:
test_roundtrip(g_parsed, "nq")

nq: matches the original.


### 2.4 `trig` (alias `trig12`)

**TriG 1.2: Turtle plus `GRAPH` blocks.** Turtle's own shorthand syntax, extended so a single document can hold multiple named graphs. `g_parsed` is one graph, so there's only one block here, named `ex:main`. See the [datasets guide](02a-graphs-datasets.ipynb) for multiple graphs in one document. `StarLayerDataset.serialize()` defaults to TriG; a graph view (`StarLayerGraph`) with an identifier writes its triples inside that graph's block, as below. The default writer is flat; `pretty=True` gives the grouped layout.

In [9]:
print(g_parsed.serialize(format="trig"))

@version "1.2" .
@prefix rdf: <http://www.w3.org/1999/02/22-rdf-syntax-ns#> .
@prefix xsd: <http://www.w3.org/2001/XMLSchema#> .
@prefix ex: <http://example.org/> .
ex:main {
	ex:AuditSystem ex:reported <<( ex:alice ex:worksFor ex:AcmeCorp )>> .
	ex:stmt2 rdf:reifies <<( ex:alice ex:worksWith ex:SalesTeam )>> .
	ex:alice ex:worksWith ex:SalesTeam .
	ex:stmt1 ex:source ex:AuditSystem ;
		ex:confidence "0.9" ;
		rdf:reifies <<( ex:alice ex:mentions ex:GlobalTech )>> .
	ex:alice ex:mentions ex:GlobalTech .
	_:f376e376c533a70f781b4ee2787a5608 ex:source ex:CRM ;
		ex:since "2020" ;
		rdf:reifies <<( ex:alice ex:likes ex:ProductABC )>> .
	ex:alice ex:likes ex:ProductABC .
	ex:id ex:reportedDate "2026-01-01"^^xsd:date ;
		ex:source ex:HRSystem ;
		rdf:reifies <<( ex:alice ex:worksFor ex:AcmeCorp )>> .
	_:d2c2c3da891c9b220417c7b76f84b424 ex:source ex:HRSystem ;
		rdf:reifies <<( ex:alice ex:worksFor ex:AcmeCorp )>> .
	ex:claim ex:confidence "high" ;
		ex:source ex:HRSystem ;
		rdf:reifies <<( 

In [10]:
test_roundtrip(g_parsed, "trig")

trig: matches the original.


### 2.5 TriX (dropped)

The old class wrote `trix12`, an XML serialization of a dataset following Apache Jena's convention. TriX is not a W3C specification and Oxigraph has no TriX support, so the format was dropped by decision with no replacement: asking for it raises `ValueError`. Use `trig` or `nq` for a dataset in one document, or `xml` for RDF/XML.

### 2.6 `xml` (alias `rdfxml12`)

**RDF/XML 1.2: the verbose, W3C-original XML serialization.** `rdf:Description` groups triples by subject; a triple term is carried via `rdf:parseType="Triple"`.

In [11]:
print(g_parsed.serialize(format="xml"))

<?xml version="1.0" encoding="UTF-8"?>
<rdf:RDF xmlns:ex="http://example.org/" xmlns:rdf="http://www.w3.org/1999/02/22-rdf-syntax-ns#" xmlns:xsd="http://www.w3.org/2001/XMLSchema#" xmlns:its="http://www.w3.org/2005/11/its">
	<rdf:Description rdf:about="http://example.org/AuditSystem">
		<ex:reported rdf:version="1.2" rdf:parseType="Triple">
			<rdf:Description rdf:about="http://example.org/alice">
				<ex:worksFor rdf:resource="http://example.org/AcmeCorp"/>
			</rdf:Description>
		</ex:reported>
	</rdf:Description>
	<rdf:Description rdf:about="http://example.org/stmt2">
		<rdf:reifies rdf:version="1.2" rdf:parseType="Triple">
			<rdf:Description rdf:about="http://example.org/alice">
				<ex:worksWith rdf:resource="http://example.org/SalesTeam"/>
			</rdf:Description>
		</rdf:reifies>
	</rdf:Description>
	<rdf:Description rdf:about="http://example.org/alice">
		<ex:worksWith rdf:resource="http://example.org/SalesTeam"/>
	</rdf:Description>
	<rdf:Description rdf:about="http://example.or

In [12]:
test_roundtrip(g_parsed, "xml")

xml: matches the original.


### 2.7 `json-ld`

**JSON-LD** works for plain RDF. Oxigraph cannot yet write RDF 1.2 content (triple terms, reifiers) as JSON-LD, so the reference graph, which has them, is refused with an `OSError`; a graph without them serializes and round-trips. If you need triple terms in a document, use one of the formats above.

In [13]:
# the reference graph has triple terms: JSON-LD cannot carry them
try:
    g_parsed.serialize(format="json-ld")
except OSError as error:
    print("OSError:", error)

# a plain graph is fine
plain = StarLayerGraph()
plain.bind("ex", EX)
plain.parse(data='@prefix ex: <http://example.org/> . ex:alice ex:worksFor ex:AcmeCorp ; ex:name "Alice" .', format="turtle")
text = plain.serialize(format="json-ld")
print(text)
back = StarLayerGraph()
back.parse(data=text, format="json-ld")
print("json-ld:", "matches the original." if isomorphic(plain, back) else "differs from the original.")

OSError: JSON-LD does not support RDF 1.2 yet
[{"@id":"http://example.org/alice","http://example.org/name":[{"@value":"Alice"}],"http://example.org/worksFor":[{"@id":"http://example.org/AcmeCorp"}]}]
json-ld: matches the original.
